In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd "/content/drive/MyDrive/Colab Notebooks/MachineLearning/diabetes_readmission_patient_reliability_extension/diabetes_readmission_replication_train_test"

/content/drive/MyDrive/Colab Notebooks/MachineLearning/diabetes_readmission_patient_reliability_extension/diabetes_readmission_replication_train_test


In [4]:
!ls


ASSUMPTIONS.md	      README.md				src
config		      references			tests
data		      requirements.txt			Untitled0.ipynb
experiments	      results				validate_extension.py
PAPER_METHODOLOGY.md  results_original_encounter_level


In [5]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 MB 3.3 MB/s eta 0:00:00


In [ ]:
import os

print(os.path.exists("data/raw/diabetic_data.csv"))
print(os.path.exists("data/raw/IDS_mapping.csv"))

True
True


In [6]:
import sklearn
import pandas
import numpy
import xgboost
import joblib

print("sklearn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("xgboost:", xgboost.__version__)

sklearn: 1.8.0
pandas: 2.2.3
numpy: 2.1.3
xgboost: 3.1.2


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

print("StratifiedGroupKFold: OK")

StratifiedGroupKFold: OK


In [ ]:
!find . -iname "diabetic_data.csv"

./data/raw/diabetic_data.csv


In [ ]:
!python -m pytest -q

.......                                                                  [100%]
7 passed in 17.50s


In [ ]:
# 1. Patient-level split
!python -m src.run_replication --stage split

Train split: (81347, 51)
Test split:  (20419, 51)
Saved: /content/drive/MyDrive/Colab Notebooks/MachineLearning/diabetes_readmission_patient_reliability_extension/diabetes_readmission_replication_train_test/data/splits/train.csv
Saved: /content/drive/MyDrive/Colab Notebooks/MachineLearning/diabetes_readmission_patient_reliability_extension/diabetes_readmission_replication_train_test/data/splits/test.csv


In [ ]:
# 2. Preprocessing
!python -m src.run_replication --stage preprocess

Saved train-only engineered data: /content/drive/MyDrive/Colab Notebooks/MachineLearning/diabetes_readmission_patient_reliability_extension/diabetes_readmission_replication_train_test/data/processed/train_final_dataset.csv
Engineered shape: (81347, 54)
Predictor count before one-hot: 53
Positive class: 9072; negative class: 72275


In [ ]:
# 3. Patient-level nested CV
!python -m src.run_replication --stage nested_cv

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

In [ ]:
# 4. Final model + Platt calibration + Cbase + Ccorr
!python -m src.run_replication --stage final_model

{
  "selected_threshold": 0.13,
  "auc": 0.6661707317668346,
  "pr_auc": 0.2091621101977898,
  "brier": 0.09501136172748388,
  "f1": 0.2746350899818278,
  "sensitivity": 0.5164241622574955,
  "specificity": 0.7182843306814252,
  "ppv": 0.18705581729617504,
  "npv": 0.9220795367755458,
  "threshold": 0.13,
  "correctness_correctness_brier": 0.09501126060509757,
  "correctness_correctness_log_loss": 0.3308364745253502,
  "correctness_correctness_ece": 0.0008566749174997597,
  "correctness_actual_accuracy": 0.6957724316815617,
  "correctness_mean_correctness_confidence": 0.6957571060710201
}


In [ ]:
# 5. Bootstrap
!python -m src.run_replication --stage bootstrap

{
  "auc": {
    "lower_2.5": 0.6599202616278886,
    "upper_97.5": 0.6719007625830214,
    "mean": 0.6661261182508336
  },
  "brier": {
    "lower_2.5": 0.09344499809942215,
    "upper_97.5": 0.09665539088208186,
    "mean": 0.09500947005809288
  },
  "risk_ratio": {
    "lower_2.5": 2.3051091422163053,
    "upper_97.5": 2.4909086004977437,
    "mean": 2.3998898670922046
  }
}


In [ ]:
# 6. DCA + CIC + clinical utility
!python -m src.run_replication --stage clinical

Clinical utility outputs generated from TRAIN OOF predictions; threshold=0.130.


In [ ]:
# 7. SHAP + stability
!python -m src.run_replication --stage shap

 93%|=================== | 931/1000 [00:12<00:00]                        feature  mean_abs_shap
        number_inpatient       0.036236
discharge_group_Transfer       0.031061
  acute_care_utilization       0.022869
 discharge_group_Expired       0.013450
    discharge_group_Home       0.012086
 hospital_pressure_index       0.011973
        number_diagnoses       0.009783
                     age       0.009544
          diabetesMed_No       0.007791
 total_healthcare_visits       0.007677
         num_medications       0.005277
        time_in_hospital       0.004144
         has_circulatory       0.003985
         has_respiratory       0.003983
            has_diabetes       0.003059
        metformin_Steady       0.002746
      num_lab_procedures       0.002689
              insulin_No       0.002296
            metformin_No       0.001874
            insulin_Down       0.001680


In [ ]:
# 8. Final untouched test evaluation
!python -m src.test_model

100%|===================| 20393/20419 [04:13<00:00]       Held-out TEST evaluation complete.
{
  "n_test": 20419,
  "positive_rate": 0.11190557813800872,
  "roc_auc": 0.6679951510985928,
  "pr_auc": 0.21264010051358612,
  "brier_score": 0.09516085980292319,
  "accuracy": 0.6880846270630295,
  "precision": 0.18350898946063235,
  "recall": 0.5181619256017506,
  "f1": 0.27103124642325743,
  "threshold": 0.13,
  "mean_correctness_confidence": 0.690046892083996,
  "correctness_brier": 0.09515773898936325,
  "correctness_log_loss": 0.33101783277014724,
  "correctness_ece": 0.0031419658900362565,
  "prediction_accuracy": 0.6880846270630295,
  "tn": 12866,
  "fp": 5268,
  "fn": 1101,
  "tp": 1184,
  "specificity": 0.7094959744127054,
  "npv": 0.9211713324264338
}
Selected threshold: 0.130
Mean correctness confidence: 0.690
Correctness ECE: 0.0031
Per-record predictions: results/tables/test_predictions.csv
Per-record SHAP: results/tables/test_record_level_shap.csv


In [ ]:
import pandas as pd

df = pd.read_csv("results/tables/test_predictions.csv")

print(df.columns.tolist())
print(df.head())

['original_row_id', 'y_true', 'actual_label', 'predicted_probability', 'predicted_class', 'predicted_label', 'confidence', 'correctness_confidence', 'uncertainty', 'prediction_correct', 'threshold']
   original_row_id  y_true                   actual_label  \
0                9       0  not_readmitted_within_30_days   
1               10       0  not_readmitted_within_30_days   
2               15       0  not_readmitted_within_30_days   
3               16       1      readmitted_within_30_days   
4               19       0  not_readmitted_within_30_days   

   predicted_probability  predicted_class                predicted_label  \
0               0.128600                0  not_readmitted_within_30_days   
1               0.099499                0  not_readmitted_within_30_days   
2               0.130354                1      readmitted_within_30_days   
3               0.075980                0  not_readmitted_within_30_days   
4               0.086382                0  not_readmit

In [ ]:
print(df[[
    "confidence",
    "correctness_confidence",
    "uncertainty",
    "prediction_correct"
]].describe())

         confidence  correctness_confidence   uncertainty  prediction_correct
count  20419.000000            20419.000000  20419.000000        20419.000000
mean       0.690462                0.690047      0.309953            0.688085
std        0.343691                0.344063      0.344063            0.463287
min        0.130002                0.129209      0.001939            0.000000
25%        0.207032                0.205982      0.068158            0.000000
50%        0.906418                0.906215      0.093785            1.000000
75%        0.931970                0.931842      0.794018            1.000000
max        0.998057                0.998061      0.870791            1.000000


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("results/tables/test_predictions.csv")

# Difference between the two confidence estimates
df["confidence_difference"] = (
    df["correctness_confidence"] - df["confidence"]
)

print("Mean Cbase:", df["confidence"].mean())
print("Mean Ccorr:", df["correctness_confidence"].mean())
print("Mean actual accuracy:", df["prediction_correct"].mean())

print("\nMean Ccorr - Cbase:",
      df["confidence_difference"].mean())

print("\nMean absolute difference:",
      df["confidence_difference"].abs().mean())

print("\nCorrelation:",
      df["confidence"].corr(df["correctness_confidence"]))

print("\nDifference summary:")
print(df["confidence_difference"].describe())

Mean Cbase: 0.6904618151518107
Mean Ccorr: 0.690046892083996
Mean actual accuracy: 0.6880846270630295

Mean Ccorr - Cbase: -0.0004149230678146467

Mean absolute difference: 0.0004150738085343656

Correlation: 0.9999999493731272

Difference summary:
count    20419.000000
mean        -0.000415
std          0.000387
min         -0.001287
25%         -0.000858
50%         -0.000203
75%         -0.000127
max          0.000005
Name: confidence_difference, dtype: float64


In [ ]:
print(
    df.loc[
        df["confidence_difference"].abs() > 0.05,
        [
            "confidence",
            "correctness_confidence",
            "uncertainty",
            "prediction_correct"
        ]
    ].head(20)
)

print(
    "Records with |Ccorr-Cbase| > 0.05:",
    (df["confidence_difference"].abs() > 0.05).sum()
)

Empty DataFrame
Columns: [confidence, correctness_confidence, uncertainty, prediction_correct]
Index: []
Records with |Ccorr-Cbase| > 0.05: 0


In [9]:
import os

print("Patient-level table files:")
for f in os.listdir("results/tables"):
    if "patient" in f.lower():
        print(f)

Patient-level table files:
patient_group_validation.csv
xgb_nested_patient_group_checks.csv
